# V15-B2 — Native Qwen2.5-VL-7B / formulaires standardisés

Expérience reconstruite sur la philosophie du pipeline 7B rapide d'origine :
**classification courte → prompt explicite par type → JSON → validation Python → retry ciblé**.

Principes :
- pas d'aliases `f01/f02`;
- pas de format positionnel;
- pas de logique 27B transplantée;
- les 99 champs canoniques sont conservés dans le RAW final;
- le VLM ne remplit que les champs utiles de la page;
- aucune correction OCR lettre→chiffre;
- les champs absents/illisibles restent `null`;
- recovery uniquement sur champs critiques ou structure incohérente.


In [ ]:
import os, re, json, time, math
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import fitz
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq

VERSION = "GENERIC_V15_B2_NATIVE_QWEN2_5_VL_7B_STANDARDIZED"
MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-Instruct/main"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_DIR = Path("/mnt/data/domiciliation_in")
OUTPUT_DIR = Path("/mnt/data/document_pipeline/V15_B2")
RAW_DIR = OUTPUT_DIR / "01_extraction_raw" / "json_dossiers"
DEBUG_DIR = OUTPUT_DIR / "debug_pages"
for p in (RAW_DIR, DEBUG_DIR): p.mkdir(parents=True, exist_ok=True)

IMAGE_MAX_SIZE = 1800
RECOVERY_MAX_SIZE = 2400
PDF_ZOOM = 2.5
MAX_CLASSIFY_TOKENS = 45

print("Version :", VERSION)
print("Device  :", DEVICE)
print("Model   :", MODEL_PATH)


In [ ]:
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval().to(DEVICE)
print(f"Modèle chargé en {time.time()-t0:.1f}s")


In [ ]:
DOM_FIELDS = [
"DOM_NOM_RAISON_SOCIAL_CLIENT","DOM_COMPTE_LOCAL","DOM_ADRESSE_CLIENT","DOM_AGENCE_DOMICILIATAIRE",
"DOM_NUMERO_CONTRAT","DOM_DUREE_CONTRAT_MOIS","DOM_DATE_DEBUT_CONTRAT","DOM_DATE_FIN_CONTRAT",
"DOM_NOM_RAISON_SOCIAL_EMPLOYEUR","DOM_ADRESSE_EMPLOYEUR","DOM_SALAIRE_NET_MENSUEL",
"DOM_PART_TRANSFERABLE","DOM_TAUX_TRANSFERABLE","DOM_MONTANT_TOTAL_DOMICILIE","DOM_DATE_SIGNATURE"]

CTR_FIELDS = [
"CTR_REFERENCE_DOCUMENT","CTR_TYPE","CTR_EMPLOYEUR","CTR_ACTIVITE_EMPLOYEUR","CTR_DUREE_MOIS",
"CTR_DATE_DEBUT_CONTRAT","CTR_POSTE","CTR_NOM_PRENOM_TRAVAILLEUR","CTR_PERE_NOM_PRENOM",
"CTR_MERE_NOM_PRENOM","CTR_NATIONALITE","CTR_DATE_NAISSANCE","CTR_LIEU_PAYS_NAISSANCE",
"CTR_ADRESSE_ALGERIE","CTR_QUALIFICATION","CTR_NUMERO_PERMIS_TRAVAIL","CTR_DATE_DELIVRANCE_PERMIS",
"CTR_DATE_DEBUT_VALIDITE_PERMIS","CTR_DATE_FIN_VALIDITE_PERMIS","CTR_SALAIRE_BRUT","CTR_SALAIRE_NET",
"CTR_AFFILIATION_SS","CTR_NUMERO_EMPLOYEUR","CTR_DATE_SIGNATURE","CTR_REFERENCE_DOMICILIATION",
"CTR_SIGNATURE_TRAVAILLEUR_PRESENTE","CTR_SIGNATURE_EMPLOYEUR_PRESENTE","CTR_CACHET_EMPLOYEUR_PRESENT"]

CTS_FIELDS = [
"CTS_REFERENCE_DOCUMENT","CTS_SAP_ID","CTS_EMPLOYEUR","CTS_ACTIVITE_EMPLOYEUR","CTS_DUREE_MOIS",
"CTS_DATE_DEBUT_CONTRAT","CTS_POSTE","CTS_NOM_PRENOM_TRAVAILLEUR","CTS_PERE_NOM_PRENOM",
"CTS_MERE_NOM_PRENOM","CTS_NATIONALITE","CTS_DATE_NAISSANCE","CTS_LIEU_PAYS_NAISSANCE",
"CTS_ADRESSE_ALGERIE","CTS_QUALIFICATION","CTS_NUMERO_PERMIS_TRAVAIL","CTS_DATE_DELIVRANCE_PERMIS",
"CTS_DATE_DEBUT_VALIDITE_PERMIS","CTS_DATE_FIN_VALIDITE_PERMIS","CTS_LIGNE_SALAIRE_BRUTE",
"CTS_SALAIRE_NET","CTS_SALAIRE_NET_ANCIEN","CTS_MENTION_AU_LIEU_DE_PRESENTE","CTS_PART_TRANSFERABLE",
"CTS_PART_PAYABLE_DZD","CTS_NUMERO_SS_PAYS_ORIGINE","CTS_NUMERO_SS_ALGERIE","CTS_DATE_DOCUMENT",
"CTS_SIGNATURE_TRAVAILLEUR_PRESENTE","CTS_SIGNATURE_EMPLOYEUR_PRESENTE","CTS_CACHET_EMPLOYEUR_PRESENT",
"CTS_VISA_INSPECTION_TRAVAIL_PRESENT"]

TTR_FIELDS = [
"TTR_NUMERO_PERMIS","TTR_NUMERO_MANUSCRIT","TTR_POSTE","TTR_DUREE","TTR_DATE_DEBUT","TTR_DATE_FIN",
"TTR_LIEU_TRAVAIL","TTR_EMPLOYEUR","TTR_ADRESSE_EMPLOYEUR","TTR_FAIT_A","TTR_DATE_DELIVRANCE",
"TTR_NOM","TTR_PRENOM","TTR_DATE_NAISSANCE","TTR_LIEU_NAISSANCE","TTR_PAYS","TTR_NATIONALITE",
"TTR_QUALIFICATION","TTR_DATE_ENTREE_ALGERIE","TTR_PHOTO_PRESENTE","TTR_CACHET_PRESENT"]

PTR_FIELDS = ["PTR_NUMERO_SERIE","PTR_WILAYA","PTR_CACHET_DIRECTION_EMPLOI_PRESENT"]
ALL_FIELDS = DOM_FIELDS + CTR_FIELDS + CTS_FIELDS + TTR_FIELDS + PTR_FIELDS
assert len(ALL_FIELDS) == 99 and len(set(ALL_FIELDS)) == 99
print("Schéma canonique :", len(ALL_FIELDS), "champs")


In [ ]:
def resize(img, max_side=IMAGE_MAX_SIZE):
    w,h = img.size
    if max(w,h) <= max_side: return img
    r = max_side/max(w,h)
    return img.resize((max(1,round(w*r)), max(1,round(h*r))), Image.LANCZOS)

def render_page(doc, i, max_side=IMAGE_MAX_SIZE):
    pix = doc.load_page(i).get_pixmap(matrix=fitz.Matrix(PDF_ZOOM, PDF_ZOOM), alpha=False)
    return resize(Image.frombytes("RGB",[pix.width,pix.height],pix.samples), max_side)

def ask(prompt, image, max_new_tokens):
    messages=[{"role":"user","content":[
        {"type":"image","image":image},{"type":"text","text":prompt}]}]
    txt=processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs=processor(text=[txt], images=[image], return_tensors="pt")
    inputs={k:(v.to(DEVICE) if hasattr(v,"to") else v) for k,v in inputs.items()}
    t=time.time()
    with torch.no_grad():
        out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,
                           repetition_penalty=1.0,use_cache=True,
                           pad_token_id=processor.tokenizer.eos_token_id,
                           eos_token_id=processor.tokenizer.eos_token_id)
    gen=out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(gen,skip_special_tokens=True,clean_up_tokenization_spaces=False), time.time()-t, int(gen.numel())

def parse_json(text):
    if not text: return {}
    text=text.strip()
    try: return json.loads(text)
    except: pass
    a,b=text.find("{"),text.rfind("}")
    if a>=0 and b>a:
        try: return json.loads(text[a:b+1])
        except: pass
    return {}


In [ ]:
PROMPT_TYPE = """Lis le titre principal de cette page et retourne uniquement :
{"type":"ENGAGEMENT_DOMICILIATION"}
ou {"type":"CONTRAT_TRAVAIL"}
ou {"type":"CONTRAT_SPECIFIQUE"}
ou {"type":"TITRE_TRAVAIL"}
ou {"type":"AUTRE"}.

Repères :
ENGAGEMENT_DOMICILIATION = formulaire "ENGAGEMENT DE DOMICILIATION".
CONTRAT_TRAVAIL = "CONTRAT DE TRAVAIL A DUREE DETERMINEE".
CONTRAT_SPECIFIQUE = "CONTRAT DE TRAVAIL SPECIFIQUE A LA MAIN D'OEUVRE ETRANGERE".
TITRE_TRAVAIL = permis/titre de travail avec identité/photo et bloc durée DU/AU.
Base-toi sur le document visible. Aucun commentaire."""

PROMPT_DOM = """Lis ce formulaire standard ENGAGEMENT DE DOMICILIATION.
Retourne uniquement ce JSON :
{
"DOM_NOM_RAISON_SOCIAL_CLIENT":null,"DOM_COMPTE_LOCAL":null,"DOM_ADRESSE_CLIENT":null,
"DOM_AGENCE_DOMICILIATAIRE":null,"DOM_NUMERO_CONTRAT":null,"DOM_DUREE_CONTRAT_MOIS":null,
"DOM_DATE_DEBUT_CONTRAT":null,"DOM_DATE_FIN_CONTRAT":null,
"DOM_NOM_RAISON_SOCIAL_EMPLOYEUR":null,"DOM_ADRESSE_EMPLOYEUR":null,
"DOM_SALAIRE_NET_MENSUEL":null,"DOM_PART_TRANSFERABLE":null,"DOM_TAUX_TRANSFERABLE":null,
"DOM_MONTANT_TOTAL_DOMICILIE":null,"DOM_DATE_SIGNATURE":null
}
Lis uniquement les valeurs visibles. Ne déduis rien. Absent/illisible = null.
IMPORTANT : dans la section opération, toute la couche des valeurs peut être décalée
verticalement ensemble vers le haut OU le bas. Ne fais pas d'association par alignement fixe.
Utilise l'ordre structurel et le type des valeurs :
numéro contrat -> durée en mois -> date début -> date fin -> employeur -> adresse employeur
-> salaire net -> part transférable -> pourcentage -> montant domicilié.
Le montant domicilié est souvent vide. Une date ne peut pas être un numéro de contrat.
Retourne uniquement le JSON."""

PROMPT_CTR = """Lis ce CONTRAT DE TRAVAIL standard. Retourne uniquement ce JSON :
{
"CTR_REFERENCE_DOCUMENT":null,"CTR_TYPE":null,"CTR_EMPLOYEUR":null,"CTR_ACTIVITE_EMPLOYEUR":null,
"CTR_DUREE_MOIS":null,"CTR_DATE_DEBUT_CONTRAT":null,"CTR_POSTE":null,
"CTR_NOM_PRENOM_TRAVAILLEUR":null,"CTR_PERE_NOM_PRENOM":null,"CTR_MERE_NOM_PRENOM":null,
"CTR_NATIONALITE":null,"CTR_DATE_NAISSANCE":null,"CTR_LIEU_PAYS_NAISSANCE":null,
"CTR_ADRESSE_ALGERIE":null,"CTR_QUALIFICATION":null,"CTR_NUMERO_PERMIS_TRAVAIL":null,
"CTR_DATE_DELIVRANCE_PERMIS":null,"CTR_DATE_DEBUT_VALIDITE_PERMIS":null,
"CTR_DATE_FIN_VALIDITE_PERMIS":null,"CTR_SALAIRE_BRUT":null,"CTR_SALAIRE_NET":null,
"CTR_AFFILIATION_SS":null,"CTR_NUMERO_EMPLOYEUR":null,"CTR_DATE_SIGNATURE":null,
"CTR_REFERENCE_DOMICILIATION":null,"CTR_SIGNATURE_TRAVAILLEUR_PRESENTE":null,
"CTR_SIGNATURE_EMPLOYEUR_PRESENTE":null,"CTR_CACHET_EMPLOYEUR_PRESENT":null
}
Règles : lis seulement ce qui est visible; aucune déduction; absent/illisible=null.
Pour le permis, recopie la référence complète y compris la partie après "/".
Dates de validité : première date après "Valable du", deuxième après "au".
Salaire brut/net : recopie exactement les montants.
Les 3 champs signature/cachet sont des booléens visuels.
Retourne uniquement le JSON."""

PROMPT_CTS = """Lis ce CONTRAT DE TRAVAIL SPECIFIQUE standard. Retourne uniquement ce JSON :
{
"CTS_REFERENCE_DOCUMENT":null,"CTS_SAP_ID":null,"CTS_EMPLOYEUR":null,"CTS_ACTIVITE_EMPLOYEUR":null,
"CTS_DUREE_MOIS":null,"CTS_DATE_DEBUT_CONTRAT":null,"CTS_POSTE":null,
"CTS_NOM_PRENOM_TRAVAILLEUR":null,"CTS_PERE_NOM_PRENOM":null,"CTS_MERE_NOM_PRENOM":null,
"CTS_NATIONALITE":null,"CTS_DATE_NAISSANCE":null,"CTS_LIEU_PAYS_NAISSANCE":null,
"CTS_ADRESSE_ALGERIE":null,"CTS_QUALIFICATION":null,"CTS_NUMERO_PERMIS_TRAVAIL":null,
"CTS_DATE_DELIVRANCE_PERMIS":null,"CTS_DATE_DEBUT_VALIDITE_PERMIS":null,
"CTS_DATE_FIN_VALIDITE_PERMIS":null,"CTS_LIGNE_SALAIRE_BRUTE":null,"CTS_SALAIRE_NET":null,
"CTS_SALAIRE_NET_ANCIEN":null,"CTS_MENTION_AU_LIEU_DE_PRESENTE":null,
"CTS_PART_TRANSFERABLE":null,"CTS_PART_PAYABLE_DZD":null,"CTS_NUMERO_SS_PAYS_ORIGINE":null,
"CTS_NUMERO_SS_ALGERIE":null,"CTS_DATE_DOCUMENT":null,"CTS_SIGNATURE_TRAVAILLEUR_PRESENTE":null,
"CTS_SIGNATURE_EMPLOYEUR_PRESENTE":null,"CTS_CACHET_EMPLOYEUR_PRESENT":null,
"CTS_VISA_INSPECTION_TRAVAIL_PRESENT":null
}
Lis seulement ce qui est visible; aucune déduction; absent/illisible=null.
Permis : référence complète, y compris après "/".
Salaire : CTS_LIGNE_SALAIRE_BRUTE = ligne complète. CTS_SALAIRE_NET = premier montant.
Si "au lieu de" existe, CTS_SALAIRE_NET_ANCIEN = second montant et mention=true; sinon null/false.
Les champs signatures/cachets/visa sont des booléens visuels.
Retourne uniquement le JSON."""

PROMPT_TTR = """Lis ce TITRE/PERMIS DE TRAVAIL standard.
Retourne uniquement :
{
"TTR_NUMERO_PERMIS":null,"TTR_NOM":null,"TTR_PRENOM":null,"TTR_DATE_NAISSANCE":null,
"TTR_NATIONALITE":null,"TTR_DATE_DEBUT":null,"TTR_DATE_FIN":null
}
Règles : lis uniquement ce qui est visible; aucune déduction; illisible=null.
Le numéro de permis est la référence imprimée en haut du document : recopie-la exactement,
y compris les deux parties séparées par "/" si elles sont visibles.
ATTENTION DECALAGE : le POSTE peut prendre une ou deux lignes et toutes les valeurs du bloc
peuvent être décalées ensemble. Pour les dates, utilise la structure :
POSTE -> DUREE -> DU/première date -> AU/deuxième date -> LIEU DE TRAVAIL.
Première date après DUREE = TTR_DATE_DEBUT; deuxième = TTR_DATE_FIN.
Ne fais jamais une association par alignement horizontal fixe.
Retourne uniquement le JSON."""

PROMPTS={"ENGAGEMENT_DOMICILIATION":PROMPT_DOM,"CONTRAT_TRAVAIL":PROMPT_CTR,
         "CONTRAT_SPECIFIQUE":PROMPT_CTS,"TITRE_TRAVAIL":PROMPT_TTR}
TOKENS={"ENGAGEMENT_DOMICILIATION":420,"CONTRAT_TRAVAIL":700,
        "CONTRAT_SPECIFIQUE":800,"TITRE_TRAVAIL":260}


In [ ]:
DATE_RE=re.compile(r"\b(?:0?[1-9]|[12]\d|3[01])[./-](?:0?[1-9]|1[0-2])[./-](?:19|20)\d{2}\b")
def missing(v): return v is None or str(v).strip() in ("","null","None")
def looks_date(v): return bool(DATE_RE.search(str(v or "")))
def digits(v): return re.sub(r"\D","",str(v or ""))

CRITICAL={
"ENGAGEMENT_DOMICILIATION":["DOM_NUMERO_CONTRAT","DOM_DUREE_CONTRAT_MOIS","DOM_DATE_DEBUT_CONTRAT",
 "DOM_DATE_FIN_CONTRAT","DOM_SALAIRE_NET_MENSUEL","DOM_PART_TRANSFERABLE","DOM_TAUX_TRANSFERABLE"],
"CONTRAT_TRAVAIL":["CTR_NOM_PRENOM_TRAVAILLEUR","CTR_NUMERO_PERMIS_TRAVAIL","CTR_DATE_DEBUT_VALIDITE_PERMIS",
 "CTR_DATE_FIN_VALIDITE_PERMIS","CTR_SALAIRE_NET"],
"CONTRAT_SPECIFIQUE":["CTS_NOM_PRENOM_TRAVAILLEUR","CTS_NUMERO_PERMIS_TRAVAIL",
 "CTS_DATE_DEBUT_VALIDITE_PERMIS","CTS_DATE_FIN_VALIDITE_PERMIS","CTS_SALAIRE_NET","CTS_PART_TRANSFERABLE"],
"TITRE_TRAVAIL":["TTR_NUMERO_PERMIS","TTR_NOM","TTR_PRENOM","TTR_DATE_NAISSANCE",
 "TTR_NATIONALITE","TTR_DATE_DEBUT","TTR_DATE_FIN"]}

def issues(dt,d):
    x=[]
    for f in CRITICAL.get(dt,[]):
        if missing(d.get(f)): x.append(f+":MISSING")
    date_fields=[f for f in CRITICAL.get(dt,[]) if "DATE_" in f]
    for f in date_fields:
        if not missing(d.get(f)) and not looks_date(d.get(f)): x.append(f+":NOT_DATE")
    if dt=="ENGAGEMENT_DOMICILIATION":
        if not missing(d.get("DOM_DUREE_CONTRAT_MOIS")) and not digits(d.get("DOM_DUREE_CONTRAT_MOIS")):
            x.append("DOM_DUREE_CONTRAT_MOIS:NOT_NUMERIC")
        if not missing(d.get("DOM_TAUX_TRANSFERABLE")) and not digits(d.get("DOM_TAUX_TRANSFERABLE")):
            x.append("DOM_TAUX_TRANSFERABLE:NOT_PERCENT")
    return sorted(set(x))

def retry_critical(dt,d,img):
    bad=issues(dt,d)
    fields=sorted(set(z.split(":")[0] for z in bad))
    if not fields: return d,[],0.0,0
    schema="{"+",".join(json.dumps(f)+":null" for f in fields)+"}"
    special=""
    if dt=="ENGAGEMENT_DOMICILIATION":
        special="La couche des valeurs peut être décalée globalement vers le haut ou le bas; utilise ordre et type."
    elif dt=="TITRE_TRAVAIL":
        special="Dates : POSTE -> DUREE -> DU/date début -> AU/date fin -> LIEU. Poste peut avoir 1 ou 2 lignes."
    p=f"""Relis cette page {dt}. Corrige UNIQUEMENT ces champs : {", ".join(fields)}.
{special}
Aucune déduction. Illisible=null. Retourne uniquement ce JSON : {schema}"""
    r,sec,tok=ask(p,img,max_new_tokens=min(300,80+55*len(fields)))
    q=parse_json(r)
    for f in fields:
        if f in q and not missing(q[f]): d[f]=q[f]
    return d,issues(dt,d),sec,tok


In [ ]:
def process_pdf(pdf_path, verbose=True):
    doc=fitz.open(pdf_path)
    canonical={f:None for f in ALL_FIELDS}
    pages_meta=[]
    total_tokens=0; total_model=0.0
    seen=set()
    t_all=time.time()

    for i in range(len(doc)):
        img=render_page(doc,i,IMAGE_MAX_SIZE)
        img.save(DEBUG_DIR/f"{Path(pdf_path).stem}_p{i+1}.png")

        raw,sec,tok=ask(PROMPT_TYPE,img,MAX_CLASSIFY_TOKENS)
        total_model+=sec; total_tokens+=tok
        dt=parse_json(raw).get("type","AUTRE")
        if verbose: print(f"Page {i+1}: {dt} | classification {sec:.1f}s")

        if dt not in PROMPTS:
            pages_meta.append({"page":i+1,"type":dt,"status":"IGNOREE"})
            continue

        raw,sec,tok=ask(PROMPTS[dt],img,TOKENS[dt])
        total_model+=sec; total_tokens+=tok
        data=parse_json(raw)
        before=issues(dt,data)

        retry_sec=0; retry_tok=0
        if before:
            # Un seul recovery ciblé, image 2400 seulement en cas d'anomalie.
            img_hd=render_page(doc,i,RECOVERY_MAX_SIZE)
            data,after,retry_sec,retry_tok=retry_critical(dt,data,img_hd)
            total_model+=retry_sec; total_tokens+=retry_tok
        else:
            after=[]

        allowed = DOM_FIELDS if dt=="ENGAGEMENT_DOMICILIATION" else CTR_FIELDS if dt=="CONTRAT_TRAVAIL" else CTS_FIELDS if dt=="CONTRAT_SPECIFIQUE" else TTR_FIELDS
        for f in allowed:
            if f in data: canonical[f]=data[f]

        pages_meta.append({"page":i+1,"type":dt,"initial_issues":before,"final_issues":after,
                           "extract_s":round(sec,3),"retry_s":round(retry_sec,3),
                           "status":"OK" if not after else "A_REVOIR"})
        seen.add(dt)
        if verbose:
            print(f"  extraction {sec:.1f}s | retry {retry_sec:.1f}s | problèmes finaux: {after or 'aucun'}")

    doc.close()
    elapsed=time.time()-t_all
    result={
        "schema":"DOM_EXTRACTION_V1",
        "pipeline_version":VERSION,
        "source_file":Path(pdf_path).name,
        "processed_at":datetime.now().isoformat(timespec="seconds"),
        "fields":canonical,
        "pages":pages_meta,
        "metrics":{"elapsed_s":round(elapsed,3),"model_generate_s":round(total_model,3),
                   "generated_tokens":total_tokens,"vlm_calls":sum(2+(1 if p.get("retry_s",0)>0 else 0)
                   for p in pages_meta if p.get("type") in PROMPTS)}
    }
    out=RAW_DIR/(Path(pdf_path).stem+".json")
    out.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
    if verbose:
        print(f"TOTAL {elapsed:.1f}s | model {total_model:.1f}s | tokens {total_tokens} | {out}")
    return result


In [ ]:
pdfs=sorted(INPUT_DIR.glob("*.pdf"))
print("PDF trouvés :",len(pdfs))
results=[]
for p in pdfs:
    print("\n"+"="*72)
    results.append(process_pdf(p,verbose=True))

summary=[]
for r in results:
    summary.append({
        "fichier":r["source_file"],
        "elapsed_s":r["metrics"]["elapsed_s"],
        "model_generate_s":r["metrics"]["model_generate_s"],
        "generated_tokens":r["metrics"]["generated_tokens"],
        "vlm_calls":r["metrics"]["vlm_calls"],
        "pages_a_revoir":sum(1 for p in r["pages"] if p.get("status")=="A_REVOIR")
    })
pd.DataFrame(summary)
